<a href="https://colab.research.google.com/github/Rais-Ataullov/BigData/blob/lab1/L1_Apache_Spark_Tasks.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Решите следующие задачи для данных велопарковок Сан-Франциско (trips.csv, stations.csv):

In [27]:
from pyspark import SparkContext, SparkConf
from pyspark.sql import SparkSession
import pyspark.sql as sql

conf = SparkConf().setAppName("L1_Apache_Spark_Tasks").setMaster('local[*]')

sc = SparkContext(conf=conf)
spark = SparkSession(sc)

In [28]:
tripData = spark.read\
.option("header", True)\
.option("inferSchema", True)\
.option("timestampFormat", 'M/d/y H:m')\
.csv("trips.csv").dropna()

stationData = spark.read\
.option("header", True)\
.option("inferSchema", True)\
.option("timestampFormat", 'M/d/y')\
.csv("stations.csv").dropna()

In [29]:
tripData.show(10)

+----+--------+-------------------+--------------------+----------------+-------------------+--------------------+--------------+-------+-----------------+--------+
|  id|duration|         start_date|  start_station_name|start_station_id|           end_date|    end_station_name|end_station_id|bike_id|subscription_type|zip_code|
+----+--------+-------------------+--------------------+----------------+-------------------+--------------------+--------------+-------+-----------------+--------+
|4130|      71|2013-08-29 10:16:00|Mountain View Cit...|              27|2013-08-29 10:17:00|Mountain View Cit...|            27|     48|       Subscriber|   97214|
|4251|      77|2013-08-29 11:29:00|  San Jose City Hall|              10|2013-08-29 11:30:00|  San Jose City Hall|            10|     26|       Subscriber|   95060|
|4299|      83|2013-08-29 12:02:00|South Van Ness at...|              66|2013-08-29 12:04:00|      Market at 10th|            67|    319|       Subscriber|   94103|
|4927|    

In [30]:
stationData.show(10)

+---+--------------------+------------------+-------------------+----------+--------+-------------------+
| id|                name|               lat|               long|dock_count|    city|  installation_date|
+---+--------------------+------------------+-------------------+----------+--------+-------------------+
|  2|San Jose Diridon ...|         37.329732|-121.90178200000001|        27|San Jose|2013-08-06 00:00:00|
|  3|San Jose Civic Ce...|         37.330698|        -121.888979|        15|San Jose|2013-08-05 00:00:00|
|  4|Santa Clara at Al...|         37.333988|        -121.894902|        11|San Jose|2013-08-06 00:00:00|
|  5|    Adobe on Almaden|         37.331415|          -121.8932|        19|San Jose|2013-08-05 00:00:00|
|  6|    San Pedro Square|37.336721000000004|        -121.894074|        15|San Jose|2013-08-07 00:00:00|
|  7|Paseo de San Antonio|         37.333798|-121.88694299999999|        15|San Jose|2013-08-07 00:00:00|
|  8| San Salvador at 1st|         37.330165|-

In [31]:
stationData.createOrReplaceTempView("stations")
tripData.createOrReplaceTempView("trips")

1. Найти велосипед с максимальным временем пробега.

In [32]:
total_durationData = spark.sql("""
SELECT trips.bike_id, SUM(duration) as total_duration FROM trips GROUP BY bike_id ORDER BY total_duration DESC;
""")

total_durationData.show(1)

+-------+--------------+
|bike_id|total_duration|
+-------+--------------+
|    535|      18476949|
+-------+--------------+
only showing top 1 row


2. Найти наибольшее геодезическое расстояние между станциями.

In [33]:
max_distanceData = spark.sql("""
WITH station_pairs AS (
    SELECT
        s1.id AS station1_id,
        s1.name AS station1_name,
        s1.lat AS lat1,
        s1.long AS lon1,
        s2.id AS station2_id,
        s2.name AS station2_name,
        s2.lat AS lat2,
        s2.long AS lon2
    FROM stations s1
    CROSS JOIN stations s2
    WHERE s1.id < s2.id
),

station_distances AS (
    SELECT
        station1_id,
        station1_name,
        station2_id,
        station2_name,
        -- Радиус Земли в км = 6371
        6371 * 2 * ASIN(
            SQRT(
                POWER(SIN(RADIANS(lat2 - lat1) / 2), 2) +
                COS(RADIANS(lat1)) * COS(RADIANS(lat2)) *
                POWER(SIN(RADIANS(lon2 - lon1) / 2), 2)
            )
        ) AS distance_km
    FROM station_pairs
)

SELECT
    station1_name,
    station2_name,
    ROUND(distance_km, 2) AS distance_km
FROM station_distances
WHERE distance_km IS NOT NULL
ORDER BY distance_km DESC
LIMIT 1;
""")

max_distanceData.show(truncate=False)

+--------------------------+----------------------+-----------+
|station1_name             |station2_name         |distance_km|
+--------------------------+----------------------+-----------+
|SJSU - San Salvador at 9th|Embarcadero at Sansome|69.92      |
+--------------------------+----------------------+-----------+



3. Найти путь велосипеда с максимальным временем пробега через станции.

In [34]:
total_durationData.createOrReplaceTempView("total_duration")

max_duration_bike_way = spark.sql("""
-- 1. Находим велосипед с максимальным суммарным пробегом
WITH bike_total_duration AS (
    SELECT
        bike_id,
        SUM(duration) AS total_duration
    FROM trips
    WHERE duration IS NOT NULL
    GROUP BY bike_id
),
max_bike AS (
    SELECT bike_id
    FROM bike_total_duration
    ORDER BY total_duration DESC
    LIMIT 1
),
-- 2. Получаем все поездки этого велосипеда в хронологическом порядке
bike_trips AS (
    SELECT
        t.start_date,
        t.end_date,
        t.start_station_name,
        t.end_station_name,
        t.duration,
        ROW_NUMBER() OVER (ORDER BY t.start_date) AS trip_number
    FROM trips t
    INNER JOIN max_bike m ON t.bike_id = m.bike_id
    WHERE t.duration IS NOT NULL
),
-- 3. Формируем путь (только уникальные переходы между станциями)
path_with_duplicates AS (
    -- Начальная станция первой поездки
    SELECT
        1 AS stop_order,
        start_station_name AS station_name,
        'START' AS stop_type,
        start_date AS time,
        1 AS trip_id
    FROM bike_trips
    WHERE trip_number = 1

    UNION ALL

    -- Конечные станции и начала следующих поездок
    SELECT
        ROW_NUMBER() OVER (ORDER BY time) + 1 AS stop_order,
        station_name,
        stop_type,
        time,
        trip_id
    FROM (
        -- Конец каждой поездки
        SELECT
            end_date AS time,
            end_station_name AS station_name,
            'END' AS stop_type,
            trip_number AS trip_id
        FROM bike_trips

        UNION ALL

        -- Начало следующей поездки (кроме первой)
        SELECT
            start_date AS time,
            start_station_name AS station_name,
            'START' AS stop_type,
            trip_number AS trip_id
        FROM bike_trips
        WHERE trip_number > 1
    ) t
)
-- 4. Убираем последовательные дубликаты станций
SELECT
    ROW_NUMBER() OVER (ORDER BY time) AS stop_order,
    station_name,
    stop_type,
    trip_id,
    time
FROM (
    SELECT
        *,
        LAG(station_name) OVER (ORDER BY time) AS prev_station
    FROM path_with_duplicates
) t
WHERE station_name != prev_station OR prev_station IS NULL
ORDER BY time;
""")

max_duration_bike_way.show(truncate=False)

+----------+---------------------------------------------+---------+-------+-------------------+
|stop_order|station_name                                 |stop_type|trip_id|time               |
+----------+---------------------------------------------+---------+-------+-------------------+
|1         |Post at Kearney                              |START    |1      |2013-08-29 19:32:00|
|2         |San Francisco Caltrain (Townsend at 4th)     |END      |1      |2013-08-29 19:53:00|
|3         |San Francisco Caltrain 2 (330 Townsend)      |END      |2      |2013-08-29 21:45:00|
|4         |Market at Sansome                            |END      |3      |2013-08-30 08:54:00|
|5         |2nd at South Park                            |END      |4      |2013-08-30 09:19:00|
|6         |2nd at Townsend                              |START    |5      |2013-09-01 12:58:00|
|7         |Davis at Jackson                             |END      |5      |2013-09-01 13:26:00|
|8         |San Francisco City

4. Найти количество велосипедов в системе.

In [35]:
bikes_count = spark.sql("""
SELECT COUNT(DISTINCT bike_id) AS bike_count FROM trips;
""")

bikes_count.show()

+----------+
|bike_count|
+----------+
|       700|
+----------+



5. Найти пользователей потративших на поездки более 3 часов (10800 сек).

In [36]:
users_with_trips_more_3_hours = spark.sql("""
SELECT
    zip_code,
    subscription_type,
    SUM(duration) AS total_duration_seconds
FROM trips
GROUP BY zip_code, subscription_type
HAVING SUM(duration) > 3 * 3600
""")

users_with_trips_more_3_hours.show()

+--------+-----------------+----------------------+
|zip_code|subscription_type|total_duration_seconds|
+--------+-----------------+----------------------+
|   95125|       Subscriber|                590806|
|   95008|         Customer|                758377|
|    2465|         Customer|                 18241|
|   94619|       Subscriber|                404485|
|   94502|         Customer|                 62657|
|   92869|         Customer|                 12270|
|   49518|         Customer|                 33839|
|   29464|         Customer|                104615|
|   80220|         Customer|                 94992|
|      55|         Customer|                853637|
|    3141|         Customer|                 14423|
|   30307|         Customer|                 25457|
|   94035|         Customer|                165289|
|   94010|       Subscriber|               3539415|
|   90210|         Customer|                690278|
|   19444|         Customer|                 14868|
|   95118|  

In [37]:
sc.stop()